# 示例: GIS流域划分与参数分区工作流

**相关脚本:**
- `examples/create_gis_data.py`
- `examples/generate_parameter_zones.py`
- `examples/plot_zones.py`

## 目的
此示例用于演示项目的第二部分功能：基于GIS的自动化流域处理。它将三个独立的脚本组合成一个完整的工作流，展示了如何：
1.  使用 `whitebox-tools` 这一强大的地理空间分析引擎，对DEM进行一系列标准的水文分析。
2.  从DEM数据中自动提取河网，并划分出所有子流域。
3.  将栅格格式的子流域结果转换为矢量多边形（Shapefile）。
4.  利用 `geopandas` 进行空间叠加分析，将子流域图层与土地利用、土壤类型图层进行合并。
5.  根据每个子流域内占主导地位的土地利用和土壤类型，为其创建一个唯一的、可用于水文模型的**参数分区ID** (`zone_id`)。
6.  将最终结果进行可视化，以供检查。

## 第1部分: 创建演示用的GIS数据

**脚本:** `examples/create_gis_data.py`

为了让示例可以独立运行，我们首先生成一套合成的GIS数据，包括一个DEM、一个土地利用Shapefile和一个土壤类型Shapefile。在实际应用中，您会使用自己项目的真实数据。

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
import geopandas as gpd
from shapely.geometry import LineString, Polygon
import os

def create_synthetic_dem():
    x = np.linspace(-5, 5, 200)
    y = np.linspace(-5, 5, 200)
    xx, yy = np.meshgrid(x, y)
    dem_data = np.abs(xx) + (yy * -2)
    dem_data = (1000 - dem_data).astype(np.float32)
    transform = from_origin(west=-123.0, north=49.0, xsize=0.01, ysize=0.01)
    crs = "EPSG:4326"
    output_file = '../gis_data/dem.tif'
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with rasterio.open(
        output_file, 'w', driver='GTiff',
        height=dem_data.shape[0], width=dem_data.shape[1],
        count=1, dtype=dem_data.dtype, crs=crs, transform=transform
    ) as dst:
        dst.write(dem_data, 1)
    print(f"Created {output_file}")

def create_synthetic_land_use():
    poly1 = Polygon([(-123.5, 48.75), (-121.5, 48.75), (-121.5, 49.0), (-123.5, 49.0)])
    poly2 = Polygon([(-123.5, 49.0), (-121.5, 49.0), (-121.5, 49.25), (-123.5, 49.25)])
    gdf = gpd.GeoDataFrame(
        {'land_use': ['Forest', 'Urban']}, geometry=[poly1, poly2], crs="EPSG:4326"
    )
    output_file = '../gis_data/land_use.shp'
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    gdf.to_file(output_file)
    print(f"Created {output_file}")

def create_synthetic_soil():
    poly1 = Polygon([(-123.5, 48.75), (-122.5, 48.75), (-122.5, 49.25), (-123.5, 49.25)])
    poly2 = Polygon([(-122.5, 48.75), (-121.5, 48.75), (-121.5, 49.25), (-122.5, 49.25)])
    gdf = gpd.GeoDataFrame(
        {'soil_type': ['Clay', 'Sand']}, geometry=[poly1, poly2], crs="EPSG:4326"
    )
    output_file = '../gis_data/soil.shp'
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    gdf.to_file(output_file)
    print(f"Created {output_file}")

create_synthetic_dem()
create_synthetic_land_use()
create_synthetic_soil()
print("\n所有演示用的GIS数据已创建。")

## 第2部分: 生成参数分区

**脚本:** `examples/generate_parameter_zones.py`

这是工作流的核心。我们按顺序执行以下步骤：
1.  **DEM预处理**: 对DEM进行填洼，以消除数字高程模型中的伪洼地。
2.  **水流方向和汇流累积量计算**: 使用D8算法计算水流方向和每个栅格单元的汇流累积量。
3.  **河网提取和子流域划分**: 根据汇流累积量阈值提取河网，并基于此划分亚流域。
4.  **栅格转矢量**: 将栅格格式的子流域地图转换为矢量的Shapefile文件，以便进行后续的空间分析。
5.  **空间叠加和分区**: 使用`geopandas`将子流域与土地利用和土壤图层进行叠加，计算每个子流域的主要土地利用和土壤类型，并据此生成一个组合的`zone_id`。

In [ ]:
from whitebox.whitebox_tools import WhiteboxTools

wbt = WhiteboxTools()
wbt.verbose = False # Set to True to see detailed logs

# --- 路径设置 ---
work_dir = os.path.abspath('../')
temp_dir = os.path.join(work_dir, "temp_gis")
results_dir = os.path.join(work_dir, "examples/results") # Save results to the main results folder
os.makedirs(temp_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# --- 文件路径 ---
dem_file = os.path.join(work_dir, "gis_data/dem.tif")
land_use_file = os.path.join(work_dir, "gis_data/land_use.shp")
soil_file = os.path.join(work_dir, "gis_data/soil.shp")
filled_dem = os.path.join(temp_dir, "filled_dem.tif")
d8_pointer = os.path.join(temp_dir, "d8_pointer.tif")
flow_acc = os.path.join(temp_dir, "flow_acc.tif")
streams = os.path.join(temp_dir, "streams.tif")
subbasins_raster = os.path.join(temp_dir, "subbasins.tif")
subbasins_vector = os.path.join(results_dir, "subbasins_with_zones.shp")

# --- 执行工作流 ---
print("Step 1: DEM预处理...")
wbt.fill_depressions(dem_file, filled_dem)
wbt.d8_pointer(filled_dem, d8_pointer)
wbt.d8_flow_accumulation(filled_dem, flow_acc, out_type='cells')

print("Step 2: 提取河网和划分亚流域...")
wbt.extract_streams(flow_acc, streams, threshold=500.0, zero_background=True)
wbt.subbasins(d8_pointer, streams, subbasins_raster)

print("Step 3: 栅格转矢量...")
wbt.raster_to_vector_polygons(subbasins_raster, subbasins_vector)

print("Step 4: 叠加分析并分配参数分区ID...")
subbasins_gdf = gpd.read_file(subbasins_vector)
land_use_gdf = gpd.read_file(land_use_file)
soil_gdf = gpd.read_file(soil_file)

projected_crs = "EPSG:3857"
subbasins_proj = subbasins_gdf.to_crs(projected_crs)
land_use_proj = land_use_gdf.to_crs(projected_crs)
soil_proj = soil_gdf.to_crs(projected_crs)

overlay_lu = gpd.overlay(subbasins_proj, land_use_proj, how='intersection')
overlay_lu['area'] = overlay_lu.geometry.area
dominant_lu = overlay_lu.loc[overlay_lu.groupby('VALUE')['area'].idxmax()].set_index('VALUE')[['land_use']]

overlay_soil = gpd.overlay(subbasins_proj, soil_proj, how='intersection')
overlay_soil['area'] = overlay_soil.geometry.area
dominant_soil = overlay_soil.loc[overlay_soil.groupby('VALUE')['area'].idxmax()].set_index('VALUE')[['soil_type']]

subbasins_gdf = subbasins_gdf.join(dominant_lu, on='VALUE').join(dominant_soil, on='VALUE')
subbasins_gdf['zone_id'] = subbasins_gdf['land_use'].fillna('Unknown') + '-' + subbasins_gdf['soil_type'].fillna('Unknown')

subbasins_gdf.to_file(subbasins_vector, driver='ESRI Shapefile', overwrite=True)

print(f"\n处理完成。最终带有分区的子流域文件已保存到: {subbasins_vector}")
display(subbasins_gdf.head())

## 第3部分: 可视化结果

**脚本:** `examples/plot_zones.py`

最后，我们创建一个地图来直观地验证我们的分区结果。我们将DEM作为底图，然后将我们生成的子流域Shapefile叠加在上面，并根据`zone_id`对每个子流域进行着色。

In [ ]:
import matplotlib.pyplot as plt
from rasterio.plot import show as show_raster

subbasins_file = os.path.join(results_dir, "subbasins_with_zones.shp")
dem_file = os.path.join(work_dir, "gis_data/dem.tif")
output_plot = os.path.join(results_dir, "parameter_zones_map.png")

subbasins_gdf_plot = gpd.read_file(subbasins_file)
dem_raster = rasterio.open(dem_file)

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

show_raster(dem_raster, ax=ax, cmap='gray', title="Sub-basins with Parameter Zones")

subbasins_gdf_plot.plot(
    column='zone_id',
    ax=ax,
    legend=True,
    legend_kwds={'title': "Parameter Zones", 'bbox_to_anchor': (1.25, 1)},
    cmap='tab20',
    alpha=0.7
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()

plt.savefig(output_plot)
print(f"验证地图已保存到 {output_plot}")
plt.show()